In [1]:
import matplotlib
import matplotlib.pyplot as plt
from spicelib import RawRead
import numpy as np

import plotly.io as pio
pio.renderers.default = "vscode"
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [ ]:
# Functions

def mean_without_outliers(x, z=4.0):
    x = np.asarray(x)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan

    med = np.median(x)
    mad = np.median(np.abs(x - med))

    if mad == 0:
        return float(np.mean(x))

    modified_z = 0.6745 * (x - med) / mad
    mask = np.abs(modified_z) < z

    return float(np.mean(x[mask]))

def estimate_frequency(time, signal):
    time = np.asarray(time)
    signal = np.asarray(signal)

    # Remove NaNs
    mask = np.isfinite(time) & np.isfinite(signal)
    time = time[mask]
    signal = signal[mask]

    if len(time) < 5:
        return np.nan

    # Automatic thresholds (robust)
    low = np.percentile(signal, 10)
    high = np.percentile(signal, 90)

    if high - low <= 0:
        return np.nan

    threshold = (low + high) / 2

    crossings = []

    for i in range(1, len(signal)):
        if signal[i-1] < threshold and signal[i] >= threshold:
            # linear interpolation for accurate crossing
            dt = time[i] - time[i-1]
            dv = signal[i] - signal[i-1]
            if dv != 0:
                t_cross = time[i-1] + dt * (threshold - signal[i-1]) / dv
            else:
                t_cross = time[i]
            crossings.append(t_cross)

    crossings = np.array(crossings)

    if len(crossings) < 2:
        return np.nan

    periods = np.diff(crossings)

    # remove period outliers
    T = mean_without_outliers(periods)

    if not np.isfinite(T) or T <= 0:
        return np.nan

    return 1.0 / T

def eng(value, unit="", sig=3):
    if value is None or not np.isfinite(value):
        return "—"
    prefixes = [
        (1e12, "T"), (1e9, "G"), (1e6, "M"), (1e3, "k"),
        (1.0, ""), (1e-3, "m"), (1e-6, "µ"), (1e-9, "n"), (1e-12, "p"),
    ]
    a = abs(value)
    for scale, p in prefixes:
        if a >= scale:
            return f"{value/scale:.{sig}g} {p}{unit}"
    return f"{value:.{sig}g} {unit}"

def print_metrics_from_results(results, avg_unit="V"):
    def eng(value, unit="", sig=3):
        if value is None or not np.isfinite(value):
            return "—"
        prefixes = [
            (1e12, "T"), (1e9, "G"), (1e6, "M"), (1e3, "k"),
            (1.0, ""), (1e-3, "m"), (1e-6, "µ"), (1e-9, "n"), (1e-12, "p"),
        ]
        a = abs(value)
        for scale, p in prefixes:
            if a >= scale:
                return f"{value/scale:.{sig}g} {p}{unit}"
        return f"{value:.{sig}g} {unit}"

    name_w = max(len(r["signal"]) for r in results + [{"signal": "Signal"}])
    f_w = max(len("Frequency"), max(len(eng(r["frequency_Hz"], "Hz")) for r in results))
    m_w = max(len("Average"), max(len(eng(r["avg_no_outliers"], avg_unit)) for r in results))

    print("┌" + "─"*(name_w+2) + "┬" + "─"*(f_w+2) + "┬" + "─"*(m_w+2) + "┐")
    print(f"│ {'Signal':<{name_w}} │ {'Frequency':<{f_w}} │ {'Average':<{m_w}} │")
    print("├" + "─"*(name_w+2) + "┼" + "─"*(f_w+2) + "┼" + "─"*(m_w+2) + "┤")

    for r in results:
        print(
            f"│ {r['signal']:<{name_w}} │ "
            f"{eng(r['frequency_Hz'],'Hz'):<{f_w}} │ "
            f"{eng(r['avg_no_outliers'],avg_unit):<{m_w}} │"
        )

    print("└" + "─"*(name_w+2) + "┴" + "─"*(f_w+2) + "┴" + "─"*(m_w+2) + "┘")


In [ ]:
Ids_array = []
Vgs_array = []

m_values = {
    0: "1",
    1: "2",
    2: "4",
    3: "8",
}

ng_values = {
    0: "1",
    1: "2",
    2: "4",
    3: "8",
}

In [ ]:
# neuron_tb FINAL
# Load the raw file
raw = RawRead("./simulations/neuron_tb.raw")

# See what signals are available 
print("Traces:", raw.get_trace_names())

t = raw.get_trace("time").get_wave()     # same as 
t_ms = t * 1000  # Convert from seconds to milliseconds


# Example: plot vmem and vreq
vmem = raw.get_trace("v(vmem)").get_wave()
vref = raw.get_trace("v(xneuron.xref.vref)").get_wave()
vCapAdapt = raw.get_trace("v(capadap)").get_wave()

vreq = raw.get_trace("v(REQ)").get_wave()
vack = raw.get_trace("v(ACK)").get_wave()

v_I_thr = raw.get_trace("v(I_thr)").get_wave()
v_I_lk = raw.get_trace("v(I_lk)").get_wave()

v_Iref = raw.get_trace("v(Iref)").get_wave()

v_I_adapt = raw.get_trace("v(I_adapt)").get_wave()
v_I_thrahp = raw.get_trace("v(I_thrahp)").get_wave()
v_I_lkahp = raw.get_trace("v(I_lkahp)").get_wave()


fig = go.Figure()

fig.add_trace(go.Scatter(x=t_ms, y=vmem, name="Vmem"))
fig.add_trace(go.Scatter(x=t_ms, y=vref, name="Vref"))
fig.add_trace(go.Scatter(x=t_ms, y=vCapAdapt, name="V(CapAdapt)"))
fig.add_trace(go.Scatter(x=t_ms, y=vreq, name="Vreq"))
fig.add_trace(go.Scatter(x=t_ms, y=vack, name="Vack"))
fig.add_trace(go.Scatter(x=t_ms, y=v_I_thr, name="V(I_thr)"))
fig.add_trace(go.Scatter(x=t_ms, y=v_I_lk, name="V(I_lk)"))
fig.add_trace(go.Scatter(x=t_ms, y=v_Iref, name="V(Iref)"))
fig.add_trace(go.Scatter(x=t_ms, y=v_I_adapt, name="V(I_adapt)"))
fig.add_trace(go.Scatter(x=t_ms, y=v_I_thrahp, name="V(I_thrahp)"))
fig.add_trace(go.Scatter(x=t_ms, y=v_I_lkahp, name="V(I_lkahp)"))


fig.update_layout(
    title="neuron_tb - Final schematic",
    xaxis_title="Time (ms)",
    yaxis_title="Voltage (V)",
    template="plotly_white"
)

fig.show()


In [ ]:
# neuron_T_final -  DPI voltages
# Load the raw file
raw = RawRead("./simulations/neuron_T_final.raw")

# See what signals are available
print("Traces:", raw.get_trace_names())

t = raw.get_trace("time").get_wave()     # same as 
t_ms = t * 1000  # Convert from seconds to milliseconds


# Example: plot vmem and vreq
vmem = raw.get_trace("v(vmem)").get_wave()
vreq = raw.get_trace("v(vreqb)").get_wave()
ivin = raw.get_trace("i(vin)").get_wave()
icapcr = raw.get_trace("i(vcapcr)").get_wave()
in_steal = raw.get_trace("i(vin_steal)").get_wave()

# ileak = raw.get_trace("i(vleak)").get_wave()
vack = raw.get_trace("v(vackb)").get_wave()
vref = raw.get_trace("v(vref)").get_wave() # capacitor voltage
vcurcomp = raw.get_trace("v(vcurcomp)").get_wave()
vcap_adapt = raw.get_trace("v(vcap_adapt)").get_wave()

# DPI 1
v_lk = raw.get_trace("v(i_lk)").get_wave()
v_thr = raw.get_trace("v(i_thr)").get_wave()

# DPI 2
v_lkahp = raw.get_trace("v(i_lkahp)").get_wave()
v_thrahp = raw.get_trace("v(i_thrahp)").get_wave()

# Refractory period
v_ref = raw.get_trace("v(iref)").get_wave()

# Adaptation
v_adapt = raw.get_trace("v(i_adapt)").get_wave()


fig = go.Figure()

fig.add_trace(go.Scatter(x=t_ms, y=vmem, name="Vmem"))
fig.add_trace(go.Scatter(x=t_ms, y=vreq, name="Vreq"))
fig.add_trace(go.Scatter(x=t_ms, y=ivin, name="Ivin"))
# fig.add_trace(go.Scatter(x=t_ms, y=icapcr, name="Icapcr"))
# fig.add_trace(go.Scatter(x=t_ms, y=ivmemfeed, name="IVmemfeed"))
# fig.add_trace(go.Scatter(x=t_ms, y=ileak, name="Ileak"))
fig.add_trace(go.Scatter(x=t_ms, y=vack, name="Vack"))
fig.add_trace(go.Scatter(x=t_ms, y=v_ref, name="V_Iref"))
# fig.add_trace(go.Scatter(x=t_ms, y=vcurcomp, name="Vcurcomp"))
# fig.add_trace(go.Scatter(x=t_ms, y=vcap_adapt, name="Vcap_adapt"))
# fig.add_trace(go.Scatter(x=t_ms, y=in_steal, name="In_steal"))
fig.add_trace(go.Scatter(x=t_ms, y=v_lk, name="V_lk"))
fig.add_trace(go.Scatter(x=t_ms, y=v_thr, name="V_thr"))
fig.add_trace(go.Scatter(x=t_ms, y=v_lkahp, name="V_lkahp"))
fig.add_trace(go.Scatter(x=t_ms, y=v_thrahp, name="V_thrahp"))
fig.add_trace(go.Scatter(x=t_ms, y=v_adapt, name="V_adapt"))



fig.update_layout(
    title="neuron_T - With adaptation",
    xaxis_title="Time (ms)",
    yaxis_title="Voltage (V)",
    template="plotly_white"
)

fig.show()

In [ ]:
# neuron_T_final - frequency and average voltage
raw = RawRead("./simulations/neuron_tb.raw")

# print("Available traces:")
# for name in raw.get_trace_names():
#     print("-", name)


time = raw.get_trace("time").get_wave()
time = np.array(time)

# Iterate over all traces except time
all_traces = raw.get_trace_names()
signals = [s for s in all_traces if s != "time"]

data = {}
for s in signals:
    if s in ["v(ack)", "v(req)"]:
        data[s] = -1 * (np.array(raw.get_trace(s).get_wave()) - 1.5)
    else:
        data[s] = np.array(raw.get_trace(s).get_wave())


results = []

for name, sig in data.items():
    freq = estimate_frequency(time, sig)
    avg = mean_without_outliers(sig)

    results.append({
        "signal": name,
        "frequency_Hz": freq,
        "avg_no_outliers": avg
    })


# Use it:
print_metrics_from_results(results, avg_unit="V")